# 043 — Clustering y reducción de dimensionalidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**K-means:** minimiza la inercia `J = Σ‖xᵢ − μ_c(i)‖²` alternando asignación (punto →
centroide más cercano) y actualización (centroide = media). Converge a óptimos locales:
varias corridas + k-means++. Supone clusters convexos y esféricos; exige escalar.
Elegir k: codo (heurístico) + silueta `s = (b−a)/max(a,b)`.

**Jerárquico aglomerativo:** fusiona los clusters más cercanos según linkage (single,
complete, average, Ward) y produce un dendrograma; no requiere k pero cuesta O(n²)+.
**DBSCAN:** agrupa por densidad (ε, minPts), encuentra formas arbitrarias y marca ruido.

**PCA:** con datos centrados, autovectores de la covarianza ordenados por autovalor;
proyectar a m componentes retiene fracción de varianza `Σλ₁..λ_m / Σλ`. Lineal, no
supervisado: máxima varianza ≠ máxima relevancia. t-SNE/UMAP: solo visualización local.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Iteración 1: asignación con μ=(0,1): {0 → μ₁}, {1,4,9,10,11 → μ₂};
actualización μ₁ = 0, μ₂ = 7. Iteración 2 con μ=(0,7): |4−0|=4 > |4−7|=3, así que
la asignación queda {0,1 → μ₁}, {4,9,10,11 → μ₂}; actualización μ₁ = 0.5, μ₂ = 8.5.
Iteración 3: |4−0.5| = 3.5 < |4−8.5| = 4.5 → 4 cambia a μ₁: {0,1,4}, {9,10,11};
μ₁ = 5/3 ≈ 1.667, μ₂ = 10. Iteración 4: sin cambios → convergencia.
J = (0−1.667)² + (1−1.667)² + (4−1.667)² + (9−10)² + 0 + (11−10)² ≈ 2.78 + 0.44 + 5.44 +
1 + 0 + 1 = **10.67**.

**Ejercicio 2.** (a) s = (8−2)/8 = **0.75**: bien agrupado. (b) s = (4−5)/5 = **−0.2**:
en promedio está más cerca del cluster vecino que del propio — probablemente mal asignado.
(c) s = 0: frontera indiferente entre ambos clusters.

**Ejercicio 3.** Σλ = 10. Fracciones: 0.60, 0.25, 0.10, 0.05; acumuladas: 0.60, 0.85,
0.95, 1.00. Para ≥ 90 % hacen falta **3 componentes**. Se pierde la varianza de las
direcciones descartadas (aquí el 5 % de λ₄): la parte de cada punto ortogonal al
subespacio retenido — el error de reconstrucción cuadrático medio es exactamente la suma
de los autovalores descartados.

**Ejercicio 4.** Con μ = (9,10): asignación {0,1,4,9 → μ₁} (el 9 dista 0 de μ₁ y 1 de μ₂),
{10,11 → μ₂}; μ₁ = 3.5, μ₂ = 10.5. Luego: {0,1,4 → μ₁}, {9,10,11 → μ₂}; μ₁ ≈ 1.667,
μ₂ = 10 → misma partición y J ≈ 10.67 que el ejercicio 1. En este dataset ambas semillas
convergen al mismo óptimo; con datos menos separados, semillas distintas terminan en
particiones distintas con J distinto — por eso `n_init > 1` es el default sensato.


In [ ]:
result = run_lab("ml", seed=43)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 4 — Lloyd en 1D, verificado
def lloyd_1d(xs, m1, m2, max_iter=20):
    for _ in range(max_iter):
        c1 = [x for x in xs if abs(x - m1) <= abs(x - m2)]
        c2 = [x for x in xs if abs(x - m1) > abs(x - m2)]
        n1 = sum(c1) / len(c1) if c1 else m1
        n2 = sum(c2) / len(c2) if c2 else m2
        if (n1, n2) == (m1, m2):
            break
        m1, m2 = n1, n2
    inercia = sum(min((x - m1) ** 2, (x - m2) ** 2) for x in xs)
    return (m1, m2), sorted(c1), sorted(c2), inercia

xs = [0, 1, 4, 9, 10, 11]
for seed in [(0, 1), (9, 10)]:
    (m1, m2), c1, c2, j = lloyd_1d(xs, *seed)
    print(f"semillas {seed} → centroides ({m1:.3f}, {m2:.3f})  clusters {c1} | {c2}  J={j:.2f}")


In [ ]:
# Ejercicios 2 y 3 — silueta y varianza explicada
def silueta(a, b):
    return (b - a) / max(a, b)

for a, b in [(2.0, 8.0), (5.0, 4.0), (3.0, 3.0)]:
    print(f"a={a} b={b} → s={silueta(a, b):+.2f}")

lams = [6.0, 2.5, 1.0, 0.5]
total = sum(lams)
acum = 0.0
for i, lam in enumerate(lams, 1):
    acum += lam / total
    print(f"PC{i}: {lam / total:.2f} de la varianza, acumulado {acum:.2f}")
# ≥ 90 % exige 3 componentes; lo descartado = suma de autovalores restantes (error de reconstrucción)


## Reflexión

1. El laboratorio es supervisado (umbral con etiquetas). Si perdieras las etiquetas,
   ¿qué haría k-means con k=2 sobre la misma feature y en qué caso su corte coincidiría
   con el umbral supervisado? ¿Cuándo no?
2. ¿Por qué "la inercia bajó al aumentar k" no es evidencia de mejor clustering, y qué
   métrica sí permite comparar k distintos?
3. Si aplicas PCA antes de k-means, ¿qué ganas y qué riesgo introduces cuando la
   estructura de grupos vive en direcciones de poca varianza?
